In [1]:
from dotenv import load_dotenv
from google import genai
import os
import json

load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
print("Client ready!")

Client ready!


In [2]:
with open('../prompts/writer_system_prompt.txt') as f:
    writer_prompt = f.read()
print("Writer prompt loaded!")
print(writer_prompt[:200])

Writer prompt loaded!
You are an expert Technical Product Manager with 10+ years of experience writing Product Requirements Documents (PRDs) for top tech companies.

Your job is to convert rough notes or ideas into a compl


In [3]:
rough_notes = "Build a ride-sharing app for college campuses."

response = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{writer_prompt}\n\nGenerate a complete PRD from these notes:\n{rough_notes}"
)

prd_output = response.text
print(prd_output)

# Product Requirements Document: CampusRide

## 1. Executive Summary
**CampusRide** is a hyper-local, peer-to-peer ride-sharing platform specifically engineered for college campus environments. Unlike global incumbents (Uber/Lyft), CampusRide optimizes for micro-mobility, safety within gated or semi-gated campus perimeters, and student-to-student trust. The product solves the "last-mile" commute problem for students living in off-campus housing or navigating large, sprawling university grounds during late hours when campus shuttle services are infrequent or unavailable. By integrating with university authentication systems (SSO), we ensure all users are verified students, fostering a safer, more reliable transit ecosystem.

## 2. Problem Statement
**Current Pain Points:**
*   **Safety:** General ride-sharing apps pair students with random strangers, posing potential safety risks that students are increasingly wary of.
*   **Cost:** Dynamic pricing on major apps is often prohibitive for

In [4]:
with open('../prompts/critic_system_prompt.txt') as f:
    critic_prompt = f.read()
print("Critic prompt loaded!")

Critic prompt loaded!


In [5]:
response2 = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{critic_prompt}\n\nReview this PRD:\n{prd_output}"
)

raw = response2.text.strip()
if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]

result = json.loads(raw)
print(json.dumps(result, indent=2))

{
  "status": "needs_revision",
  "score": 68,
  "missing_sections": [],
  "feedback": [
    "Lack of GTM strategy: No mention of student acquisition channels, incentivization models, or university administrative partnerships.",
    "Insurance and Liability: Missing critical regulatory and legal framework section regarding commercial insurance for personal vehicles used for ride-sharing.",
    "Resource Constraints: The timeline assumes a 6-month launch but lacks a budget, team structure, or headcount plan.",
    "Underdeveloped Error Handling: Edge cases are mentioned but lack technical robustness; for instance, what happens during server-side latency or total API failures during peak hours?",
    "Design/UI/UX: No mention of accessibility requirements or design systems, which are crucial for a university-wide application.",
    "Market Competition: Fails to address potential campus shuttle service pushback or existing student-run Facebook/Discord ride-share groups that already captur

In [6]:
# Sirisha's cell - testing critic agent
from dotenv import load_dotenv
from google import genai
import os
import json

load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

with open('../prompts/critic_system_prompt.txt') as f:
    critic_prompt = f.read()

# Paste Vedha's PRD output here
sample_prd = """# Product Requirements Document: CampusRide

**Version:** 1.0  
**Status:** Draft  
**Product Manager:** [Your Name/AI Lead PM]

---

## 1. Executive Summary
CampusRide is a hyper-local, peer-to-peer ride-sharing application specifically architected for college campuses. It connects student drivers with student passengers to provide safe, affordable, and sustainable transportation within and around university environments. Unlike general ride-sharing apps (Uber/Lyft), CampusRide leverages institutional email verification (e.g., .edu addresses) to build a closed, trusted network. The product solves the "transportation gap" for students who lack cars and face long walks or unreliable bus systems, while helping student drivers subsidize their gas and parking costs.

---

## 2. Problem Statement
**Current Pain Points:**
*   **Safety/Trust:** Students are hesitant to use public taxis or general ride-share apps with strangers who lack any campus affiliation.
*   **Economic Barriers:** Standard ride-sharing services feature dynamic pricing (surge pricing) that is often unaffordable for students on a budget.
*   **Infrastructure:** Public transportation on or near campuses is frequently infrequent or inefficient, especially during late-night hours or during inclement weather.

**Why Now:**
With the rise of "micro-mobility" and the increasing cost of campus parking permits, students are looking for community-based transit solutions. Failure to address this leaves students reliant on dangerous late-night walks or expensive, unreliable commercial alternatives.

**Impact:**
Lack of a dedicated solution leads to decreased campus safety, increased student absenteeism due to transportation issues, and a higher carbon footprint from individual car use.
...
### Closing Professional Summary
The CampusRide product is designed to fill a critical void in modern collegiate infrastructure. By prioritizing university-verified identity, route-based matching, and a community-first safety protocol, we are not just building an app; we are building a campus utility. The technical foundation—anchored by a PostGIS-enabled relational database and a reactive frontend—is built to scale. Through diligent adherence to the safety, legal, and operational milestones outlined in this PRD, the product is positioned to become the standard for safe and affordable campus transportation."""

response = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"{critic_prompt}\n\nReview this PRD:\n{sample_prd}"
)

raw = response.text.strip()
if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]

result = json.loads(raw)
print(json.dumps(result, indent=2))

{
  "status": "needs_revision",
  "score": 45,
  "missing_sections": [
    "3. Goals and Success Metrics",
    "4. Target Audience",
    "5. User Stories",
    "6. Functional Requirements",
    "7. Non-Functional Requirements",
    "8. UX/UI Requirements",
    "9. Go-to-Market/Launch Strategy",
    "10. Risks and Mitigation"
  ],
  "feedback": [
    "The PRD is critically incomplete, containing only the Executive Summary, Problem Statement, and a closing summary.",
    "Missing core requirement specifications (Functional/Non-Functional) make it impossible to assess feasibility or scope.",
    "Lack of success metrics (KPIs) means there is no way to measure the product's success or failure post-launch.",
    "Absence of a risk management section is unacceptable for a project involving physical safety and legal liabilities (e.g., insurance/liability for student drivers).",
    "User stories and user flow diagrams are absent, leaving the development team without guidance on feature intera